In [2]:
!pip uninstall -y torch torchtext torchvision torchaudio

!pip install torch==2.2.2 torchtext==0.17.2 torchvision==0.17.2

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchtext 0.18.0
Uninstalling torchtext-0.18.0:
  Successfully uninstalled torchtext-0.18.0
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━

In [1]:
!pip install torch torchtext

import math
import torch
import torch.nn as nn
import torch.optim as optim

from torchtext.datasets import WikiText2
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

In [6]:
!pip install torchdata torchtext

In [8]:
text = """
machine learning is amazing
deep learning is powerful
transformers are very powerful
attention is all you need
machine learning uses data
deep learning uses neural networks
"""

# tokenize
words = text.split()

# vocabulary
vocab = list(set(words))
word_to_idx = {w:i for i,w in enumerate(vocab)}
idx_to_word = {i:w for w,i in word_to_idx.items()}

vocab_size = len(vocab)

print("Vocabulary:", vocab)

Vocabulary: ['all', 'are', 'data', 'neural', 'uses', 'very', 'learning', 'machine', 'need', 'amazing', 'networks', 'is', 'you', 'transformers', 'attention', 'powerful', 'deep']


In [9]:
sequence_length = 3

data = []

for i in range(len(words)-sequence_length):

    seq = words[i:i+sequence_length]
    target = words[i+sequence_length]

    data.append((
        [word_to_idx[w] for w in seq],
        word_to_idx[target]
    ))

print(data[:5])

[([7, 6, 11], 9), ([6, 11, 9], 16), ([11, 9, 16], 6), ([9, 16, 6], 11), ([16, 6, 11], 15)]


In [10]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

In [11]:
class TransformerModel(nn.Module):

    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        x = self.embedding(x)

        x = self.pos_encoder(x)

        x = x.permute(1,0,2)

        x = self.transformer(x)

        x = x[-1]

        x = self.fc(x)

        return x

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerModel(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


TransformerModel(
  (embedding): Embedding(17, 64)
  (pos_encoder): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=64, out_features=17, bias=True)
)


In [14]:
epochs = 200

for epoch in range(epochs):

    total_loss = 0

    for seq, target in data:

        seq = torch.tensor(seq).unsqueeze(0).to(device)
        target = torch.tensor([target]).to(device)

        optimizer.zero_grad()

        output = model(seq)

        loss = criterion(output, target)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch} Loss {total_loss:.4f}")

Epoch 0 Loss 0.1994
Epoch 20 Loss 0.1149
Epoch 40 Loss 0.0642
Epoch 60 Loss 0.0433
Epoch 80 Loss 0.0295
Epoch 100 Loss 0.0188
Epoch 120 Loss 0.0141
Epoch 140 Loss 0.0097
Epoch 160 Loss 0.0073
Epoch 180 Loss 0.0057


In [15]:
def predict_next(words_input):

    model.eval()

    seq = [word_to_idx[w] for w in words_input]

    seq = torch.tensor(seq).unsqueeze(0).to(device)

    with torch.no_grad():

        output = model(seq)

        predicted = torch.argmax(output).item()

    print("Input:", words_input)
    print("Predicted next word:", idx_to_word[predicted])


predict_next(["machine","learning","is"])

Input: ['machine', 'learning', 'is']
Predicted next word: amazing
